# Low-Light Image Enhancement bằng CNN nhẹ — DarkIR-Edge (C=24) vs Standard U-Net
**CS2203 – Image Processing & Computer Vision.** Notebook chạy trên Google Colab (GPU T4/L4).

**Luồng chạy:** `Setup → Dữ liệu → Mô hình → Đo tốc độ → Train → Evaluate → Bảng/Hình → Hiệu năng`

**Lưu ý trung thực (nên ghi vào báo cáo):**
- `DarkIR-Edge` ở đây là bản **cài đặt lại theo mô tả DarkIR (CVPR 2025)** (EBlock có FreMLP, DBlock có Di-SpAM d=[1,4,9], SCA, Simple Gate, side-loss ở đáy) + 2 cải tiến vệ tinh (kênh $I_{max}$, Residual Gated Skip). Đây **không phải bản sao 1:1 mã nguồn gốc**, nên số tham số có thể lệch so với 1.93M trong tài liệu. Hãy in số tham số (cell 9) và dùng số đó trong báo cáo.
- Không tuyên bố tái lập SOTA. DarkIR gốc được train trên nhiều dữ liệu hơn → chỉ dùng làm mốc tham chiếu.
- Chỉ có **một** mô hình đề xuất → không kết luận được khối nào đóng góp bao nhiêu. Các cờ `use_imax / use_gate / use_fft` có sẵn nếu sau này muốn chạy thêm một run đối chứng.

In [ ]:
# ===== CELL 1: Imports + cấu hình =====
import os, sys, re, json, glob, time, math, random, hashlib, copy, zipfile, shutil, subprocess
from pathlib import Path
import numpy as np, cv2, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

IN_COLAB = 'google.colab' in sys.modules
TEST_MODE = os.environ.get('LLIE_TEST') == '1'          # chỉ dùng khi chạy thử ngoài Colab
BASE = '/content' if IN_COLAB else os.environ.get('LLIE_BASE', './llie_run')
DRIVE = '/content/drive/MyDrive/llie_project' if IN_COLAB else f'{BASE}/drive'
os.makedirs(DRIVE, exist_ok=True)

DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEV.type == 'cuda'

CFG = dict(
    zip_path   = f'{DRIVE}/llie_data_clean.zip',       # file zip dữ liệu bạn đã upload lên Drive
    data_root  = f'{BASE}/datasets',                   # nơi giải nén (SSD cục bộ của Colab)
    ckpt_dir   = f'{DRIVE}/checkpoints',
    result_dir = f'{DRIVE}/results',
    manifest   = f'{DRIVE}/split_manifest.json',
    # --- huấn luyện (khớp file 02) ---
    epochs=100, batch_size=8, patch=256, repeat=2,
    lr=2e-4, lr_min=1e-6, weight_decay=1e-4, betas=(0.9, 0.999),
    val_interval=5, ema_decay=0.999, grad_clip=1.0, num_workers=2,
    loss_w=dict(side=0.5, ssim=0.2, color=0.1),
    # --- chia dữ liệu ---
    n_train=dict(lolv2=639, huawei=561),
    n_val=dict(lolv2=50, huawei=50),
    huawei_gap=10,                                     # vùng đệm giữa train và val (tránh ảnh liền kề cùng cảnh)
    # --- các lần chạy: (tag, danh sách seed) ---
    runs=[('darkir_edge', [42, 123]), ('unet', [42])],
    unet_base=None,                                    # None = tự chọn để gần số tham số của DarkIR-Edge
    max_steps_per_epoch=None,
)
if TEST_MODE:   # cấu hình siêu nhỏ để kiểm tra code chạy được
    CFG.update(epochs=2, batch_size=2, patch=64, repeat=2, val_interval=1, num_workers=0,
               n_train=dict(lolv2=6, huawei=6), n_val=dict(lolv2=2, huawei=2), huawei_gap=1,
               runs=[('darkir_edge', [1]), ('unet', [1])], max_steps_per_epoch=3)

for d in (CFG['data_root'], CFG['ckpt_dir'], CFG['result_dir']):
    os.makedirs(d, exist_ok=True)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

try:
    import lpips
    LPIPS_OK = True
except Exception:
    if IN_COLAB:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lpips'])
    try:
        import lpips; LPIPS_OK = True
    except Exception as e:
        LPIPS_OK = False; print('WARN: không có lpips → bỏ qua LPIPS.', e)

print('torch', torch.__version__, '| device:', DEV, '| AMP:', USE_AMP, '| TEST_MODE:', TEST_MODE)

In [ ]:
# ===== CELL 2: Mount Drive + giải nén dữ liệu vào SSD cục bộ (chỉ Colab) =====
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.listdir(CFG['data_root']):
        print('Giải nén', CFG['zip_path'])
        shutil.copy(CFG['zip_path'], '/content/llie_data.zip')
        with zipfile.ZipFile('/content/llie_data.zip') as z: z.extractall(CFG['data_root'])
    print('Dữ liệu tại:', CFG['data_root'], os.listdir(CFG['data_root']))
else:
    print('Không phải Colab → bỏ qua.')

In [ ]:
# ===== CELL 3: Xem cấu trúc thư mục để điền SOURCES (cell 5) cho đúng =====
def show_tree(root, depth=3, max_items=6):
    root = Path(root)
    def rec(p, d):
        if d > depth: return
        kids = sorted(p.iterdir())
        dirs = [k for k in kids if k.is_dir()]; files = [k for k in kids if k.is_file()]
        for k in dirs: print('  ' * d + f'[D] {k.name}/'); rec(k, d + 1)
        if files: print('  ' * d + f'    ({len(files)} files) vd: ' + ', '.join(f.name for f in files[:max_items]))
    rec(root, 0)
show_tree(CFG['data_root'])

### ✏️ Việc bạn cần làm: điền đường dẫn trong `SOURCES` (cell tiếp theo)
Mỗi mục là `(thư mục ảnh tối, thư mục ảnh sáng)`, tương đối so với `data_root`. Cần đúng 5 mục:
LOLv2-Real train/test, LSRW-Huawei train/test, LSRW-Nikon test. Hàm ghép cặp ghép theo tên file sau khi bỏ chữ `low/normal/high` (vd `low00001.png` ↔ `normal00001.png`, `12.jpg` ↔ `12.jpg`) và **báo lỗi nếu số ảnh hai bên không khớp**.

In [ ]:
# ===== CELL 5: Ghép cặp ảnh + chia train/val/test + manifest (có SHA-256) =====
SOURCES = {   # <-- SỬA CHO KHỚP CẤU TRÚC THƯ MỤC THẬT CỦA BẠN (xem cell 3)
    'lolv2_real_train':  ('LOLv2-Real/Train/Low',  'LOLv2-Real/Train/Normal'),
    'lolv2_real_test':   ('LOLv2-Real/Test/Low',   'LOLv2-Real/Test/Normal'),
    'lsrw_huawei_train': ('LSRW/Huawei/Train/low', 'LSRW/Huawei/Train/high'),
    'lsrw_huawei_test':  ('LSRW/Huawei/Test/low',  'LSRW/Huawei/Test/high'),
    'lsrw_nikon_test':   ('LSRW/Nikon/Test/low',   'LSRW/Nikon/Test/high'),
}
if TEST_MODE:
    SOURCES = {k: (f'fake/{k}/low', f'fake/{k}/high') for k in SOURCES}

IMG_EXT = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
def _nat(s): return [int(t) if t.isdigit() else t for t in re.split(r'(\d+)', s)]
def _key(stem): return re.sub(r'(?i)(low|normal|high|gt)', '', stem).strip('_-. ')

def make_pairs(low_rel, high_rel):
    ld, hd = Path(CFG['data_root']) / low_rel, Path(CFG['data_root']) / high_rel
    assert ld.is_dir() and hd.is_dir(), f'Không thấy thư mục: {ld} hoặc {hd}'
    low  = {_key(p.stem): str(p) for p in ld.iterdir() if p.suffix.lower() in IMG_EXT}
    high = {_key(p.stem): str(p) for p in hd.iterdir() if p.suffix.lower() in IMG_EXT}
    diff = set(low) ^ set(high)
    assert not diff, f'{len(diff)} ảnh không có cặp, vd: {sorted(diff, key=_nat)[:5]}'
    return [(low[k], high[k]) for k in sorted(low, key=_nat)]

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''): h.update(chunk)
    return h.hexdigest()

def build_manifest():
    nv, nt, gap = CFG['n_val'], CFG['n_train'], CFG['huawei_gap']
    lol = make_pairs(*SOURCES['lolv2_real_train']);   hw = make_pairs(*SOURCES['lsrw_huawei_train'])
    # LOLv2: lấy KHỐI CUỐI làm val (ảnh liền kề thường cùng cảnh → hạn chế rò rỉ train/val)
    lol_val, lol_pool = lol[-nv['lolv2']:], lol[:-nv['lolv2']]
    if len(lol_pool) <= nt['lolv2']: lol_train = lol_pool
    else:
        idx = sorted(random.Random(0).sample(range(len(lol_pool)), nt['lolv2'])); lol_train = [lol_pool[i] for i in idx]
    # Huawei: val = khối cuối; train = lấy mẫu ngẫu nhiên (seed cố định) từ phần còn lại, chừa vùng đệm
    hw_val, hw_pool = hw[-nv['huawei']:], hw[:len(hw) - nv['huawei'] - gap]
    idx = sorted(random.Random(0).sample(range(len(hw_pool)), nt['huawei']))
    hw_train = [hw_pool[i] for i in idx]
    splits = {
        'train': {'lolv2': lol_train, 'huawei': hw_train},
        'val':   {'lolv2': lol_val,   'huawei': hw_val},
        'test':  {'lolv2': make_pairs(*SOURCES['lolv2_real_test']),
                  'huawei': make_pairs(*SOURCES['lsrw_huawei_test']),
                  'nikon':  make_pairs(*SOURCES['lsrw_nikon_test'])},
    }
    # --- kiểm tra không giao nhau ---
    sets = {(s, d): {p[0] for p in v} for s, dd in splits.items() for d, v in dd.items()}
    keys = list(sets)
    for i in range(len(keys)):
        for j in range(i + 1, len(keys)):
            assert not (sets[keys[i]] & sets[keys[j]]), f'RÒ RỈ giữa {keys[i]} và {keys[j]}'
    man = {s: {d: [[l, h, sha256(l), sha256(h)] for l, h in v] for d, v in dd.items()} for s, dd in splits.items()}
    json.dump(man, open(CFG['manifest'], 'w'), indent=1)
    return splits

if os.path.exists(CFG['manifest']):
    man = json.load(open(CFG['manifest']))
    SPLITS = {s: {d: [(x[0], x[1]) for x in v] for d, v in dd.items()} for s, dd in man.items()}
    print('Đã nạp manifest có sẵn:', CFG['manifest'])
else:
    SPLITS = build_manifest(); print('Đã tạo manifest:', CFG['manifest'])

for s, dd in SPLITS.items():
    print(f'{s:5s}', {d: len(v) for d, v in dd.items()}, '→ tổng', sum(len(v) for v in dd.values()))
# Kỳ vọng (file 02): train 639+561=1200 | val 50+50=100 | test 100+30+20=150

In [ ]:
# ===== CELL 6: Dataset + DataLoader =====
def imread_rgb(p):
    im = cv2.imread(p, cv2.IMREAD_COLOR)
    if im is None: raise IOError(f'Không đọc được {p}')
    return cv2.cvtColor(im, cv2.COLOR_BGR2RGB)

def to_tensor(x): return torch.from_numpy(np.ascontiguousarray(x)).permute(2, 0, 1).float().div_(255.)

class PairDataset(Dataset):
    # train=True: crop ngẫu nhiên patch×patch (CÙNG tọa độ cho low/high) + lật ngang/dọc; KHÔNG resize/color-jitter.
    # train=False: trả ảnh đầy đủ.
    def __init__(self, pairs, patch=256, repeat=1, train=True):
        self.pairs, self.patch, self.repeat, self.train = list(pairs), patch, repeat, train
        self.cache, nbytes = [], 0
        for lp, hp in self.pairs:
            lo, hi = imread_rgb(lp), imread_rgb(hp)
            assert lo.shape == hi.shape, f'Kích thước low/high khác nhau: {lp} {lo.shape} vs {hi.shape}'
            self.cache.append((lo, hi)); nbytes += lo.nbytes + hi.nbytes
        self.gb = nbytes / 1e9
    def __len__(self): return len(self.pairs) * (self.repeat if self.train else 1)
    def __getitem__(self, i):
        lo, hi = self.cache[i % len(self.pairs)]
        if self.train:
            H, W, _ = lo.shape; p = self.patch
            if H < p or W < p:   # ảnh nhỏ hơn patch → đệm phản xạ (hiếm)
                ph, pw = max(0, p - H), max(0, p - W)
                lo = np.pad(lo, ((0, ph), (0, pw), (0, 0)), 'reflect'); hi = np.pad(hi, ((0, ph), (0, pw), (0, 0)), 'reflect'); H, W, _ = lo.shape
            y = int(torch.randint(0, H - p + 1, (1,))); x = int(torch.randint(0, W - p + 1, (1,)))
            lo, hi = lo[y:y + p, x:x + p], hi[y:y + p, x:x + p]
            if torch.rand(1) < 0.5: lo, hi = lo[:, ::-1], hi[:, ::-1]
            if torch.rand(1) < 0.5: lo, hi = lo[::-1], hi[::-1]
        return to_tensor(lo), to_tensor(hi)

def flat(split): return [p for d in SPLITS[split].values() for p in d]
train_ds = PairDataset(flat('train'), CFG['patch'], CFG['repeat'], train=True)
val_ds   = PairDataset(flat('val'), train=False)
print(f'train_ds: {len(train_ds)} patch/epoch ({len(train_ds.pairs)} ảnh × {CFG["repeat"]}), RAM ≈ {train_ds.gb:.2f} GB | val_ds: {len(val_ds)} ảnh')

def make_train_loader(seed):
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True, drop_last=True, generator=g,
                      num_workers=CFG['num_workers'], pin_memory=DEV.type == 'cuda',
                      persistent_workers=CFG['num_workers'] > 0)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)
print('steps/epoch =', len(train_ds) // CFG['batch_size'], '| tổng steps =', CFG['epochs'] * (len(train_ds) // CFG['batch_size']))

## Mô hình
- **DarkIR-Edge**: Intro → 3 stage Encoder (EBlock có `FreMLP`) → Bottleneck (2 EBlock + 2 DBlock, nhánh `side` 8C→3 tại đáy) → 3 stage Decoder (DBlock có `Di-SpAM` d=[1,4,9]) với `GatedSkip`; đầu ra = `ending(x) + ảnh_gốc` (global residual).
- **FreMLP chạy FP32 + phasor đơn vị** (không `atan2/cos/sin`) để an toàn với AMP và ảnh kích thước bất kỳ. Mặc định `norm='ortho'` (biên độ ít phụ thuộc độ phân giải hơn `backward`).
- **Standard U-Net**: Conv-GroupNorm-ReLU, cùng độ sâu 3 lần downsample, cùng global residual, số kênh được chọn để số tham số gần DarkIR-Edge.

In [ ]:
# ===== CELL 8: Định nghĩa mô hình =====
class LayerNorm2d(nn.Module):
    def __init__(self, c, eps=1e-6):
        super().__init__(); self.w = nn.Parameter(torch.ones(c)); self.b = nn.Parameter(torch.zeros(c)); self.eps = eps
    def forward(self, x):
        x32 = x.float(); mu = x32.mean(1, keepdim=True); var = (x32 - mu).pow(2).mean(1, keepdim=True)
        y = (x32 - mu) / torch.sqrt(var + self.eps)
        return (y * self.w.view(1, -1, 1, 1) + self.b.view(1, -1, 1, 1)).to(x.dtype)

class SimpleGate(nn.Module):
    def forward(self, x): a, b = x.chunk(2, 1); return a * b

class SCA(nn.Module):      # Simplified Channel Attention: X ⊙ σ(Conv1x1(GAP(X)))  (theo công thức trong file 01)
    def __init__(self, c): super().__init__(); self.fc = nn.Conv2d(c, c, 1)
    def forward(self, x): return x * torch.sigmoid(self.fc(F.adaptive_avg_pool2d(x, 1)))

class FreMLP(nn.Module):   # chỉnh BIÊN ĐỘ trong miền Fourier, giữ nguyên PHA
    def __init__(self, c, norm='ortho', eps=1e-6):
        super().__init__(); self.norm, self.eps = norm, eps
        self.mlp = nn.Sequential(nn.Conv2d(c, c, 1), nn.LeakyReLU(0.1), nn.Conv2d(c, c, 1))
    def forward(self, x):
        H, W = x.shape[-2:]
        with torch.autocast(device_type=x.device.type, enabled=False):      # FFT luôn ở FP32
            xf = torch.fft.rfft2(x.float(), norm=self.norm)
            mag = torch.sqrt(xf.real ** 2 + xf.imag ** 2 + self.eps)
            mag_new = self.mlp(mag)
            y = torch.fft.irfft2(xf * (mag_new / mag), s=(H, W), norm=self.norm)   # xf/mag = phasor đơn vị
        return y.to(x.dtype)

class _FFN(nn.Module):
    def __init__(self, c, expand=2):
        super().__init__(); self.norm = LayerNorm2d(c); self.c1 = nn.Conv2d(c, c * expand, 1)
        self.sg = SimpleGate(); self.c2 = nn.Conv2d(c * expand // 2, c, 1); self.gamma = nn.Parameter(torch.zeros(1, c, 1, 1))
    def forward(self, x): return x + self.c2(self.sg(self.c1(self.norm(x)))) * self.gamma

class EBlock(nn.Module):   # khối Encoder: nhánh không gian (DW conv) + nhánh Fourier (FreMLP) + FFN
    def __init__(self, c, use_fft=True):
        super().__init__(); dw = c * 2
        self.norm = LayerNorm2d(c); self.c1 = nn.Conv2d(c, dw, 1); self.dw = nn.Conv2d(dw, dw, 3, padding=1, groups=dw)
        self.sg = SimpleGate(); self.sca = SCA(dw // 2); self.c3 = nn.Conv2d(dw // 2, c, 1)
        self.freq = FreMLP(c) if use_fft else None
        self.beta = nn.Parameter(torch.zeros(1, c, 1, 1)); self.ffn = _FFN(c)
    def forward(self, x):
        y = self.norm(x); s = self.c3(self.sca(self.sg(self.dw(self.c1(y)))))
        if self.freq is not None: s = s + self.freq(y)
        return self.ffn(x + s * self.beta)

class DBlock(nn.Module):   # khối Decoder: Di-SpAM = các DW conv giãn nở song song d=[1,4,9] (tổng) + SG + SCA + FFN
    def __init__(self, c, dilations=(1, 4, 9)):
        super().__init__(); dw = c * 2
        self.norm = LayerNorm2d(c); self.c1 = nn.Conv2d(c, dw, 1)
        self.branches = nn.ModuleList([nn.Conv2d(dw, dw, 3, padding=d, dilation=d, groups=dw) for d in dilations])
        self.sg = SimpleGate(); self.sca = SCA(dw // 2); self.c3 = nn.Conv2d(dw // 2, c, 1)
        self.beta = nn.Parameter(torch.zeros(1, c, 1, 1)); self.ffn = _FFN(c)
    def forward(self, x):
        y = self.c1(self.norm(x)); y = sum(b(y) for b in self.branches)
        return self.ffn(x + self.c3(self.sca(self.sg(y))) * self.beta)

class GatedSkip(nn.Module):  # Edge-LLIE: F = x + Conv1x1(x) ⊙ σ(Conv1x1(x))
    def __init__(self, c): super().__init__(); self.f = nn.Conv2d(c, c, 1); self.g = nn.Conv2d(c, c, 1)
    def forward(self, x): return x + self.f(x) * torch.sigmoid(self.g(x))

def pad8(x):
    H, W = x.shape[-2:]; ph, pw = (8 - H % 8) % 8, (8 - W % 8) % 8
    return F.pad(x, (0, pw, 0, ph), mode='reflect') if (ph or pw) else x

class DarkIREdge(nn.Module):
    def __init__(self, width=24, enc=(1, 2, 3), dec=(3, 1, 1), mid=(2, 2),
                 use_imax=True, use_gate=True, use_fft=True):
        super().__init__(); self.use_imax = use_imax
        self.intro = nn.Conv2d(4 if use_imax else 3, width, 3, padding=1)
        self.encoders, self.downs = nn.ModuleList(), nn.ModuleList(); c = width
        for n in enc:
            self.encoders.append(nn.Sequential(*[EBlock(c, use_fft) for _ in range(n)]))
            self.downs.append(nn.Conv2d(c, 2 * c, 2, 2)); c *= 2
        self.mid_enc = nn.Sequential(*[EBlock(c, use_fft) for _ in range(mid[0])])
        self.side = nn.Conv2d(c, 3, 3, padding=1)
        self.mid_dec = nn.Sequential(*[DBlock(c) for _ in range(mid[1])])
        self.ups, self.gates, self.decoders = nn.ModuleList(), nn.ModuleList(), nn.ModuleList()
        for n in dec:
            self.ups.append(nn.Sequential(nn.Conv2d(c, c * 2, 1, bias=False), nn.PixelShuffle(2))); c //= 2
            self.gates.append(GatedSkip(c) if use_gate else nn.Identity())
            self.decoders.append(nn.Sequential(*[DBlock(c) for _ in range(n)]))
        self.ending = nn.Conv2d(width, 3, 3, padding=1)
    def forward(self, x, return_side=False):
        H, W = x.shape[-2:]; x = pad8(x); inp = x
        f = torch.cat([x, x.amax(1, keepdim=True)], 1) if self.use_imax else x
        f = self.intro(f); skips = []
        for enc, down in zip(self.encoders, self.downs):
            f = enc(f); skips.append(f); f = down(f)
        f = self.mid_enc(f); side = self.side(f); f = self.mid_dec(f)
        for up, gate, dec in zip(self.ups, self.gates, self.decoders):
            f = up(f) + gate(skips.pop()); f = dec(f)
        out = (self.ending(f) + inp)[..., :H, :W]
        return (out, side) if return_side else out

def _cb(i, o, g=4):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.GroupNorm(g, o), nn.ReLU(True),
                         nn.Conv2d(o, o, 3, padding=1), nn.GroupNorm(g, o), nn.ReLU(True))

class UNetBaseline(nn.Module):   # U-Net chuẩn (plain conv), cùng độ sâu & global residual
    def __init__(self, base=24):
        super().__init__(); b = base
        self.e1, self.e2, self.e3, self.mid = _cb(3, b), _cb(b, 2 * b), _cb(2 * b, 4 * b), _cb(4 * b, 8 * b)
        self.u3, self.u2, self.u1 = nn.ConvTranspose2d(8 * b, 4 * b, 2, 2), nn.ConvTranspose2d(4 * b, 2 * b, 2, 2), nn.ConvTranspose2d(2 * b, b, 2, 2)
        self.d3, self.d2, self.d1 = _cb(8 * b, 4 * b), _cb(4 * b, 2 * b), _cb(2 * b, b)
        self.out = nn.Conv2d(b, 3, 1)
    def forward(self, x, return_side=False):
        H, W = x.shape[-2:]; x = pad8(x); inp = x
        e1 = self.e1(x); e2 = self.e2(F.max_pool2d(e1, 2)); e3 = self.e3(F.max_pool2d(e2, 2)); m = self.mid(F.max_pool2d(e3, 2))
        d3 = self.d3(torch.cat([self.u3(m), e3], 1)); d2 = self.d2(torch.cat([self.u2(d3), e2], 1)); d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        out = (self.out(d1) + inp)[..., :H, :W]
        return (out, None) if return_side else out

def n_params(m): return sum(p.numel() for p in m.parameters())

In [ ]:
# ===== CELL 9: Kiểm tra mô hình + chọn số kênh U-Net để khớp số tham số =====
edge = DarkIREdge(width=24)
p_edge = n_params(edge)
print(f'DarkIR-Edge (C=24): {p_edge:,} tham số  (tài liệu của bạn ghi 1,931,715 — lệch là bình thường vì đây là bản cài đặt lại)')
if CFG['unet_base'] is None:
    best = min(range(8, 80, 4), key=lambda b: abs(n_params(UNetBaseline(b)) - p_edge))
    CFG['unet_base'] = best
print(f'U-Net baseline: base={CFG["unet_base"]} → {n_params(UNetBaseline(CFG["unet_base"])):,} tham số')

# smoke test: forward/backward, kích thước lẻ (không chia hết cho 8)
x = torch.rand(1, 3, 61, 83)
for name, m in [('edge', edge), ('unet', UNetBaseline(CFG['unet_base']))]:
    out, side = m(x, return_side=True); out.mean().backward()
    print(name, 'out', tuple(out.shape), '| side', None if side is None else tuple(side.shape))
assert out.shape == x.shape
del edge, m

In [ ]:
# ===== CELL 10: Loss, EMA, metric =====
def _gauss_win(size=11, sigma=1.5, ch=3):
    g = torch.arange(size).float() - size // 2; g = torch.exp(-g ** 2 / (2 * sigma ** 2)); g /= g.sum()
    return (g[:, None] * g[None, :]).expand(ch, 1, size, size).contiguous()

def ssim_torch(x, y, win):
    ch = x.shape[1]; w = win.to(x.device)
    mu_x, mu_y = F.conv2d(x, w, groups=ch), F.conv2d(y, w, groups=ch)
    sxx = F.conv2d(x * x, w, groups=ch) - mu_x ** 2; syy = F.conv2d(y * y, w, groups=ch) - mu_y ** 2
    sxy = F.conv2d(x * y, w, groups=ch) - mu_x * mu_y
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    return (((2 * mu_x * mu_y + C1) * (2 * sxy + C2)) / ((mu_x ** 2 + mu_y ** 2 + C1) * (sxx + syy + C2))).mean()

class CompositeLoss(nn.Module):
    # L = L1 + w_side·L1(side, avgpool8(GT)) + w_ssim·(1-SSIM) + w_color·(1-cos_RGB)   (tính ở FP32)
    def __init__(self, w, use_color=True):
        super().__init__(); self.w, self.use_color = w, use_color; self.win = _gauss_win()
    def forward(self, pred, gt, side=None):
        pred, gt = pred.float(), gt.float(); parts = {'l1': F.l1_loss(pred, gt)}
        parts['ssim'] = self.w['ssim'] * (1 - ssim_torch(pred.clamp(0, 1), gt, self.win))
        if side is not None: parts['side'] = self.w['side'] * F.l1_loss(side.float(), F.avg_pool2d(gt, 8))
        if self.use_color:
            cos = F.cosine_similarity(pred, gt, dim=1, eps=1e-6); mask = (gt.amax(1) >= 5 / 255).float()   # bỏ pixel đen tuyệt đối
            parts['color'] = self.w['color'] * (1 - (cos * mask).sum() / mask.sum().clamp(min=1))
        return sum(parts.values()), parts

class EMA:
    def __init__(self, model, decay): self.m = copy.deepcopy(model).eval(); self.decay, self.n = decay, 0
    @torch.no_grad()
    def update(self, model):
        self.n += 1; d = min(self.decay, (1 + self.n) / (10 + self.n))
        for pe, p in zip(self.m.parameters(), model.parameters()): pe.mul_(d).add_(p.detach(), alpha=1 - d)
        for be, b in zip(self.m.buffers(), model.buffers()): be.copy_(b)

def psnr_u8(a, b):
    mse = np.mean((a.astype(np.float64) - b.astype(np.float64)) ** 2)
    return 99.0 if mse == 0 else 10 * math.log10(255.0 ** 2 / mse)

@torch.no_grad()
def predict_u8(model, img_u8):
    x = to_tensor(img_u8).unsqueeze(0).to(DEV)
    with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=USE_AMP):
        y = model(x)
    return (y.float().clamp(0, 1)[0].permute(1, 2, 0).cpu().numpy() * 255.0).round().astype(np.uint8)

@torch.no_grad()
def val_psnr(model, ds):
    model.eval(); vals = []
    for lo, hi in ds.cache: vals.append(psnr_u8(predict_u8(model, lo), hi))
    return float(np.mean(vals))

## Huấn luyện
Ngân sách bước **cố định** (100 epoch × 300 bước = 30.000), CosineAnnealing chạy hết chu kỳ, **lấy checkpoint cuối (trọng số EMA)**. Có resume: nếu Colab ngắt kết nối, chạy lại cell – nó tiếp tục từ `last.pt` và bỏ qua run đã xong.

In [ ]:
# ===== CELL 11: Hàm train (resume + EMA + AMP + clip) =====
def build(tag):
    if tag == 'darkir_edge': return DarkIREdge(width=24)
    if tag == 'unet':        return UNetBaseline(CFG['unet_base'])
    raise ValueError(tag)

def train_model(tag, seed):
    out_dir = Path(CFG['ckpt_dir']) / f'{tag}_seed{seed}'; out_dir.mkdir(parents=True, exist_ok=True)
    if (out_dir / 'final.pt').exists(): print(f'[skip] {tag} seed {seed} đã xong'); return
    set_seed(seed); model = build(tag).to(DEV); ema = EMA(model, CFG['ema_decay'])
    crit = CompositeLoss(CFG['loss_w']).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], betas=CFG['betas'], weight_decay=CFG['weight_decay'])
    loader = make_train_loader(seed)
    spe = len(loader) if CFG['max_steps_per_epoch'] is None else min(len(loader), CFG['max_steps_per_epoch'])
    total = CFG['epochs'] * spe
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total, eta_min=CFG['lr_min'])
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    history, start = [], 0
    if (out_dir / 'last.pt').exists():
        ck = torch.load(out_dir / 'last.pt', map_location=DEV)
        model.load_state_dict(ck['model']); ema.m.load_state_dict(ck['ema']); opt.load_state_dict(ck['opt'])
        sched.load_state_dict(ck['sched']); scaler.load_state_dict(ck['scaler']); history, start, ema.n = ck['history'], ck['epoch'], ck['ema_n']
        print(f'[resume] {tag} seed {seed} từ epoch {start}')
    print(f'== {tag} seed {seed}: {n_params(model):,} tham số | {spe} bước/epoch × {CFG["epochs"]} epoch ==')
    t0 = time.time()
    for ep in range(start, CFG['epochs']):
        model.train(); run = {}; n = 0
        for it, (lo, hi) in enumerate(loader):
            if it >= spe: break
            lo, hi = lo.to(DEV, non_blocking=True), hi.to(DEV, non_blocking=True)
            with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=USE_AMP):
                pred, side = model(lo, return_side=True)
            loss, parts = crit(pred, hi, side)
            if not torch.isfinite(loss): print('WARN: loss không hữu hạn → bỏ qua bước'); opt.zero_grad(set_to_none=True); continue
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); gn = torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
            scaler.step(opt); scaler.update(); sched.step(); ema.update(model)
            run['loss'] = run.get('loss', 0) + loss.item(); n += 1
            for k, v in parts.items(): run[k] = run.get(k, 0) + v.item()
        rec = {'epoch': ep + 1, 'lr': sched.get_last_lr()[0], **{k: v / max(n, 1) for k, v in run.items()}}
        if (ep + 1) % CFG['val_interval'] == 0 or ep + 1 == CFG['epochs']:
            rec['val_psnr_ema'] = val_psnr(ema.m, val_ds)
            torch.save({'model': model.state_dict(), 'ema': ema.m.state_dict(), 'opt': opt.state_dict(), 'sched': sched.state_dict(),
                        'scaler': scaler.state_dict(), 'history': history + [rec], 'epoch': ep + 1, 'ema_n': ema.n}, out_dir / 'last.pt')
        history.append(rec)
        if 'val_psnr_ema' in rec or ep == start:
            print(f'ep {ep+1:3d}/{CFG["epochs"]} loss {rec.get("loss", float("nan")):.4f} lr {rec["lr"]:.2e} '
                  f'val_PSNR(ema) {rec.get("val_psnr_ema", float("nan")):.2f} | {(time.time()-t0)/60:.1f} phút')
    pd.DataFrame(history).to_csv(out_dir / 'history.csv', index=False)
    torch.save({'ema': ema.m.state_dict(), 'tag': tag, 'seed': seed, 'cfg': {k: v for k, v in CFG.items() if k != 'runs'}}, out_dir / 'final.pt')
    print(f'✔ xong {tag} seed {seed}: {(time.time()-t0)/60:.1f} phút')

In [ ]:
# ===== CELL 12: ĐO TỐC ĐỘ trước khi train thật (200 bước đầu) → ước lượng thời gian mỗi run =====
def probe_speed(tag, steps=200):
    set_seed(0); m = build(tag).to(DEV).train(); crit = CompositeLoss(CFG['loss_w']).to(DEV)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-4); scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    it_loader = iter(make_train_loader(0)); t = None
    steps = 6 if TEST_MODE else steps
    for i in range(steps):
        if i == 3:
            if DEV.type == 'cuda': torch.cuda.synchronize()
            t = time.time()
        lo, hi = next(it_loader); lo, hi = lo.to(DEV), hi.to(DEV)
        with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=USE_AMP): pred, side = m(lo, return_side=True)
        loss, _ = crit(pred, hi, side); opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    if DEV.type == 'cuda': torch.cuda.synchronize()
    ips = (steps - 3) / (time.time() - t); total = CFG['epochs'] * (len(train_ds) // CFG['batch_size'])
    print(f'{tag}: {ips:.2f} it/s → ước lượng {total / ips / 60:.0f} phút / run ({total:,} bước)')
for tag, _ in CFG['runs']: probe_speed(tag)

In [ ]:
# ===== CELL 13: TRAIN THẬT (có thể chạy lại nhiều lần – tự resume/skip) =====
for tag, seeds in CFG['runs']:
    for s in seeds:
        train_model(tag, s)

## Đánh giá (inference trên 150 ảnh test)
PSNR/SSIM tính trên ảnh **uint8 đã làm tròn** (SSIM: Gaussian 11×11, σ=1.5, trung bình 3 kênh RGB); LPIPS (AlexNet) chỉ là *thước đo*, không phải mô hình huấn luyện. `psnr_gtmean` là **chỉ số chẩn đoán nội bộ** (căn độ sáng theo GT) — **không đưa vào bảng chính**.

In [ ]:
# ===== CELL 14: Baseline cổ điển + hàm đánh giá chung =====
from skimage.metrics import structural_similarity

def gamma_fn(img, g=2.2): return (np.power(img / 255.0, 1.0 / g) * 255).round().astype(np.uint8)

def clahe_fn(img, clip=2.0, tile=8):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB); l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile)).apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB)

def msrcr_fn(img, sigmas=(15, 80, 250), alpha=125., beta=46., G=192., off=-30., lo_p=1, hi_p=99):
    x = img.astype(np.float64) + 1.0
    msr = sum(np.log10(x) - np.log10(cv2.GaussianBlur(x, (0, 0), s)) for s in sigmas) / len(sigmas)
    cr = beta * (np.log10(alpha * x) - np.log10(x.sum(axis=2, keepdims=True)))
    out = G * (msr * cr - off)
    for c in range(3):
        lo, hi = np.percentile(out[..., c], [lo_p, hi_p]); out[..., c] = np.clip((out[..., c] - lo) / (hi - lo + 1e-6), 0, 1) * 255
    return out.round().astype(np.uint8)

LP = None
if LPIPS_OK:
    try: LP = lpips.LPIPS(net='alex', verbose=False).to(DEV).eval()
    except Exception as e: print('WARN: không khởi tạo được LPIPS (cần tải trọng số AlexNet) → bỏ qua LPIPS.', e); LPIPS_OK = False

@torch.no_grad()
def metrics(pred, gt):
    r = {'psnr': psnr_u8(pred, gt),
         'ssim': structural_similarity(pred, gt, channel_axis=2, data_range=255, gaussian_weights=True, sigma=1.5, use_sample_covariance=False)}
    s = gt.mean() / max(pred.mean(), 1e-6)
    r['psnr_gtmean'] = psnr_u8(np.clip(pred.astype(np.float64) * s, 0, 255).round().astype(np.uint8), gt)   # CHỈ CHẨN ĐOÁN
    if LP is not None:
        t = lambda a: (to_tensor(a).unsqueeze(0) * 2 - 1).to(DEV)
        r['lpips'] = float(LP(t(pred), t(gt)).item())
    return r

def eval_method(name, fn, seed=None):
    rows = []
    for domain, pairs in SPLITS['test'].items():
        for lp, hp in pairs:
            lo, hi = imread_rgb(lp), imread_rgb(hp)
            rows.append({'method': name, 'seed': seed, 'domain': domain, 'image': Path(lp).name, **metrics(fn(lo), hi)})
    return pd.DataFrame(rows)

def nn_fn(tag, seed):
    ck = torch.load(Path(CFG['ckpt_dir']) / f'{tag}_seed{seed}' / 'final.pt', map_location=DEV)
    m = build(tag).to(DEV).eval(); m.load_state_dict(ck['ema']); return lambda img: predict_u8(m, img)

In [ ]:
# ===== CELL 15: CHẠY ĐÁNH GIÁ → results.csv (mỗi dòng = 1 ảnh × 1 phương pháp × 1 seed) =====
frames = [eval_method('Input (không xử lý)', lambda x: x), eval_method('Gamma 2.2', gamma_fn),
          eval_method('CLAHE', clahe_fn), eval_method('MSRCR', msrcr_fn)]
for tag, seeds in CFG['runs']:
    for s in seeds:
        if (Path(CFG['ckpt_dir']) / f'{tag}_seed{s}' / 'final.pt').exists():
            frames.append(eval_method({'darkir_edge': 'DarkIR-Edge (ours)', 'unet': 'Standard U-Net'}[tag], nn_fn(tag, s), s))
RES = pd.concat(frames, ignore_index=True); RES.to_csv(f"{CFG['result_dir']}/results_per_image.csv", index=False)
print(RES.groupby(['method', 'seed'], dropna=False)[['psnr', 'ssim'] + (['lpips'] if LPIPS_OK else [])].mean().round(3))

In [ ]:
# ===== CELL 16: Bảng kết quả (Mean ± Std qua seed) + paired test =====
from scipy import stats
METRICS = [m for m in ['psnr', 'ssim', 'lpips'] if m in RES.columns]

def summarize(df):
    # với mỗi (method, seed): trung bình theo miền, gộp (pooled theo ảnh) và macro (trung bình 3 miền) → rồi mean±std qua seed
    rows = []
    for (meth, seed), g in df.groupby(['method', 'seed'], dropna=False):
        r = {'method': meth, 'seed': seed}
        for m in METRICS:
            dom = g.groupby('domain')[m].mean()
            for d, v in dom.items(): r[f'{m}|{d}'] = v
            r[f'{m}|pooled'] = g[m].mean(); r[f'{m}|macro'] = dom.mean()
        rows.append(r)
    per_seed = pd.DataFrame(rows)
    agg = per_seed.drop(columns='seed').groupby('method', sort=False).agg(['mean', 'std'])
    return per_seed, agg

per_seed, agg = summarize(RES)
def fmt(meth, col):
    mu, sd = agg.loc[meth, (col, 'mean')], agg.loc[meth, (col, 'std')]
    return f'{mu:.3f}' + (f' ± {sd:.3f}' if not np.isnan(sd) else '')
cols = [c for c in per_seed.columns if c not in ('method', 'seed')]
TABLE = pd.DataFrame({c: [fmt(m, c) for m in agg.index] for c in cols}, index=agg.index)
TABLE.to_csv(f"{CFG['result_dir']}/table_main.csv")
pd.set_option('display.width', 250, 'display.max_columns', 50)
print('— PSNR —'); print(TABLE[[c for c in cols if c.startswith('psnr|')]])
print('— SSIM —'); print(TABLE[[c for c in cols if c.startswith('ssim|')]])
if LPIPS_OK: print('— LPIPS (thấp hơn = tốt hơn) —'); print(TABLE[[c for c in cols if c.startswith('lpips|')]])

# --- Paired test: DarkIR-Edge vs U-Net trên từng ảnh (PSNR trung bình qua seed) ---
A, B = 'DarkIR-Edge (ours)', 'Standard U-Net'
if A in RES.method.values and B in RES.method.values:
    pa = RES[RES.method == A].groupby(['domain', 'image']).psnr.mean(); pb = RES[RES.method == B].groupby(['domain', 'image']).psnr.mean()
    d = (pa - pb).dropna().values
    rng = np.random.default_rng(0); ci = np.percentile([rng.choice(d, len(d)).mean() for _ in range(2000)], [2.5, 97.5])
    print(f'\nΔPSNR ({A} − {B}), N={len(d)}: mean={d.mean():+.3f} dB, 95% bootstrap CI [{ci[0]:+.3f}, {ci[1]:+.3f}]')
    print('paired t-test p =', f'{stats.ttest_rel(pa.loc[pb.index], pb).pvalue:.4g}', '| Wilcoxon p =', f'{stats.wilcoxon(d).pvalue:.4g}',
          '| tỉ lệ ảnh Edge > U-Net:', f'{(d > 0).mean():.0%}')
    print('Lưu ý: kiểm định này chỉ nói về quần thể ảnh test, không thay cho độ lệch giữa các seed (xem cột ± ở bảng).')

In [ ]:
# ===== CELL 17: Hình minh họa định tính + đường cong học =====
import matplotlib; import matplotlib.pyplot as plt
def pick(k=2):
    out = []
    for d, pairs in SPLITS['test'].items(): out += [(d, p) for p in pairs[:k]]
    return out
ex = pick(2); tag_seed = {t: s[0] for t, s in CFG['runs']}
fns = {'Input': lambda x: x, 'CLAHE': clahe_fn}
if 'unet' in tag_seed: fns['U-Net'] = nn_fn('unet', tag_seed['unet'])
if 'darkir_edge' in tag_seed: fns['DarkIR-Edge'] = nn_fn('darkir_edge', tag_seed['darkir_edge'])
fig, ax = plt.subplots(len(ex), len(fns) + 1, figsize=(3.2 * (len(fns) + 1), 2.4 * len(ex)))
ax = np.atleast_2d(ax)
for r, (dom, (lp, hp)) in enumerate(ex):
    lo, hi = imread_rgb(lp), imread_rgb(hp)
    for c, (n, f) in enumerate(fns.items()):
        im = f(lo); ax[r, c].imshow(im); ax[r, c].set_title(f'{n}\n{psnr_u8(im, hi):.2f} dB' if n != 'Input' else f'Input ({dom})', fontsize=8); ax[r, c].axis('off')
    ax[r, -1].imshow(hi); ax[r, -1].set_title('Ground truth', fontsize=8); ax[r, -1].axis('off')
plt.tight_layout(); plt.savefig(f"{CFG['result_dir']}/qualitative.png", dpi=130); plt.show()

# đường cong học (seed đầu tiên của mỗi mô hình)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for tag, seeds in CFG['runs']:
    h = Path(CFG['ckpt_dir']) / f'{tag}_seed{seeds[0]}' / 'history.csv'
    if h.exists():
        df = pd.read_csv(h); ax[0].plot(df.epoch, df.loss, label=tag)
        v = df.dropna(subset=['val_psnr_ema']); ax[1].plot(v.epoch, v.val_psnr_ema, marker='o', label=tag)
ax[0].set_title('Train loss'); ax[1].set_title('Val PSNR (EMA)'); [a.set_xlabel('epoch') or a.legend() for a in ax]
plt.tight_layout(); plt.savefig(f"{CFG['result_dir']}/curves.png", dpi=130); plt.show()

In [ ]:
# ===== CELL 18: Bảng hiệu năng: tham số, FLOPs, latency, bộ nhớ =====
from torch.utils.flop_counter import FlopCounterMode
rows = []
for tag, name in [('darkir_edge', 'DarkIR-Edge'), ('unet', 'Standard U-Net')]:
    m = build(tag).to(DEV).eval(); r = {'model': name, 'params(M)': round(n_params(m) / 1e6, 3)}
    with FlopCounterMode(display=False) as fc, torch.no_grad(): m(torch.rand(1, 3, 256, 256, device=DEV))
    r['GFLOPs@256² (FFT không được đếm)'] = round(fc.get_total_flops() / 1e9, 2)
    if DEV.type == 'cuda':
        x = torch.rand(1, 3, 400, 600, device=DEV)
        for prec, dt in [('fp32', None), ('fp16', torch.float16)]:
            with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16, enabled=dt is not None):
                for _ in range(10): m(x)
                torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(); t = time.time()
                for _ in range(50): m(x)
                torch.cuda.synchronize(); ms = (time.time() - t) / 50 * 1000
            r[f'latency {prec} @600×400 (ms)'] = round(ms, 1); r[f'FPS {prec}'] = round(1000 / ms, 1)
        r['peak mem (MB)'] = round(torch.cuda.max_memory_allocated() / 2 ** 20)
    rows.append(r)
EFF = pd.DataFrame(rows); EFF.to_csv(f"{CFG['result_dir']}/efficiency.csv", index=False); print(EFF.T)

## (Tùy chọn) Mốc tham chiếu: chạy checkpoint **DarkIR chính thức** trên đúng 150 ảnh test
Chỉ cần inference, không train. Cách làm: clone repo DarkIR, tải checkpoint theo hướng dẫn trong repo, tạo mô hình đúng config của họ, rồi bọc thành hàm `ảnh uint8 RGB → ảnh uint8 RGB` và gọi `eval_method`. Khung mẫu (bạn cần chỉnh đúng tên lớp/đường dẫn theo README của repo):

```python
# official = <khởi tạo mô hình DarkIR chính thức, nạp checkpoint, .eval().to(DEV)>
# def official_fn(img):
#     x = to_tensor(img).unsqueeze(0).to(DEV)
#     with torch.no_grad(): y = official(x)            # nếu mô hình trả về nhiều đầu ra, lấy đầu ra cuối
#     return (y.clamp(0,1)[0].permute(1,2,0).cpu().numpy()*255).round().astype(np.uint8)
# RES = pd.concat([RES, eval_method('DarkIR (official, tham chiếu)', official_fn)])
```
Ghi rõ trong báo cáo: DarkIR gốc được train trên nhiều dữ liệu hơn → đây là **mốc trần tham chiếu**, không phải so sánh công bằng. Chạy cell 16 lại để đưa dòng này vào bảng; nếu PSNR của nó gần số trong paper trên cùng tập test thì pipeline đánh giá của bạn đúng.

## Gợi ý đọc kết quả (cho phần "Đánh giá & hạn chế")
1. So Edge với U-Net: nhìn **cột ± theo seed** trước, rồi mới nhìn p-value. Chênh nhỏ hơn độ lệch seed ⇒ không kết luận.
2. `psnr_gtmean` ≫ `psnr` ⇒ lỗi chủ yếu là **độ sáng toàn cục** (bài toán một-nhiều: GT chỉ là một mức sáng hợp lý). Gần bằng nhau ⇒ lỗi là **chi tiết/nhiễu**. (Chỉ *chẩn đoán loại lỗi*, không quy nguyên nhân cho FFT.)
3. So miền **Nikon** (cảm biến chưa từng thấy) với Huawei/LOLv2 ⇒ nhận xét về tổng quát hóa; chỉ 20 ảnh nên nêu dưới dạng quan sát, không khẳng định mạnh.
4. Xem `qualitative.png`: tìm vùng tối phẳng còn nhiễu, ám màu, vùng cháy sáng ⇒ viết phần ca thất bại.